In [ ]:
import pandas as pd
import numpy as np
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

###Dataset loading and first overview

In [ ]:
data = pd.read_csv('quora.csv')
data.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [ ]:
print(f"{data.shape}\n")
print(f"{data.isnull().sum()}\n")
print(f"{data['is_duplicate'].value_counts()}\n")
print(f"{data['is_duplicate'].value_counts(normalize=True) * 100}\n")

(404290, 6)

id              0
qid1            0
qid2            0
question1       1
question2       2
is_duplicate    0
dtype: int64

is_duplicate
0    255027
1    149263
Name: count, dtype: int64

is_duplicate
0    63.080215
1    36.919785
Name: proportion, dtype: float64



In [ ]:
data.dropna(inplace=True)
print(data.shape)

(404287, 6)


###Text cleaning by converting to lowercase, removing stopwords and applying stemming

In [ ]:
nltk.download('stopwords')

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def txt_cleaner(text):
    """
    Text cleaning by converting to lowercase, removing custom stopwords
    and stemming.
    """

    text = str(text).lower()

    words = text.split()

    clean_word_list = [stemmer.stem(word) for word in words if word not in stop_words]

    return " ".join(clean_word_list)

data['q1_clean'] = data['question1'].apply(txt_cleaner)
data['q2_clean'] = data['question2'].apply(txt_cleaner)

display(data[['question1', 'q1_clean', 'question2', 'q2_clean']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,question1,q1_clean,question2,q2_clean
0,What is the step by step guide to invest in sh...,step step guid invest share market india?,What is the step by step guide to invest in sh...,step step guid invest share market?
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,stori kohinoor (koh-i-noor) diamond?,What would happen if the Indian government sto...,would happen indian govern stole kohinoor (koh...
2,How can I increase the speed of my internet co...,increas speed internet connect use vpn?,How can Internet speed be increased by hacking...,internet speed increas hack dns?
3,Why am I mentally very lonely? How can I solve...,mental lonely? solv it?,Find the remainder when [math]23^{24}[/math] i...,"find remaind [math]23^{24}[/math] divid 24,23?"
4,"Which one dissolve in water quikly sugar, salt...","one dissolv water quikli sugar, salt, methan c...",Which fish would survive in salt water?,fish would surviv salt water?


### Use of Jaccard similarity score to compare the similarity of the paired questions

In [ ]:
data['q1_len'] = data['q1_clean'].str.len()
data['q2_len'] = data['q2_clean'].str.len()

data['q1_wrd_cnt'] = data['q1_clean'].str.split().str.len()
data['q2_wrd_cnt'] = data['q2_clean'].str.split().str.len()

data['wrd_cnt_diff'] = abs(data['q1_wrd_cnt'] - data['q2_wrd_cnt'])

def overlapping(row):
    """
    Calculates word overlap by converting q1_clean, q2_clean into sets that
    ignore duplicates.
    Computes number of common words and produces the jaccard similarity score
    for the pair.
    """

    w1 = set(str(row['q1_clean']).split())
    w2 = set(str(row['q2_clean']).split())

    intersect = len(w1 & w2)
    union = len(w1 | w2)

    if union > 0:
      jaccard = intersect / union
    else:
      jaccard = 0

    return pd.Series([intersect, jaccard])

data[['common_words', 'jacc_sml']] = data.apply(overlapping, axis=1)

display(data[['is_duplicate', 'q1_wrd_cnt', 'q2_wrd_cnt', 'common_words', 'jacc_sml']].head())

,is_duplicate,q1_wrd_cnt,q2_wrd_cnt,common_words,jacc_sml
0,0,7,6,4.0,0.571429
1,0,4,9,2.0,0.181818
2,0,6,5,3.0,0.375000
3,0,4,5,0.0,0.000000
4,0,10,5,0.0,0.000000


### Convert the raw text into numerical feature vectors with TfidfVectorizer, including n-grams as a parameter and use of cosine similarity score

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances

tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))

tfidf.fit(data['q1_clean'].tolist() + data['q2_clean'].tolist())

q1_tfidf = tfidf.transform(data['q1_clean'])
q2_tfidf = tfidf.transform(data['q2_clean'])

data['cos_sml'] = 1 - paired_cosine_distances(q1_tfidf, q2_tfidf)

display(data[['is_duplicate', 'common_words', 'jacc_sml', 'cos_sml']].head())

,is_duplicate,common_words,jacc_sml,cos_sml
0,0,4.0,0.571429,0.974336
1,0,2.0,0.181818,0.000000
2,0,3.0,0.375000,0.699195
3,0,0.0,0.000000,0.000000
4,0,0.0,0.000000,0.657088


### Model building and results

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, f1_score, classification_report
from scipy.sparse import hstack

feature_cols = ['q1_len', 'q2_len', 'q1_wrd_cnt', 'q2_wrd_cnt',
                'wrd_cnt_diff', 'common_words', 'jacc_sml', 'cos_sml']

extra_features = data[feature_cols].values

X = hstack([q1_tfidf, q2_tfidf, extra_features])

y = data['is_duplicate']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

Logistic Regression

In [ ]:
model_lr = LogisticRegression(max_iter=3000, random_state=42, n_jobs=-1)
model_lr.fit(X_train, y_train)

y_pred = model_lr.predict(X_test)

y_pred_proba = model_lr.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
logloss = log_loss(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"F1-Score: {f1:.4f}")
print(f"Log-Loss: {logloss:.4f}")

print(classification_report(y_test, y_pred))

Accuracy: 76.69%
F1-Score: 0.6692
Log-Loss: 0.4622
              precision    recall  f1-score   support

           0       0.80      0.84      0.82     76508
           1       0.70      0.64      0.67     44779

    accuracy                           0.77    121287
   macro avg       0.75      0.74      0.74    121287
weighted avg       0.76      0.77      0.76    121287



Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
logloss_rf = log_loss(y_test, y_pred_proba_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print(f"Accuracy: {accuracy_rf * 100:.2f}%")
print(f"F1-Score: {f1_rf:.4f}")
print(f"Log-Loss: {logloss_rf:.4f}")

print(classification_report(y_test, y_pred_rf))

Accuracy: 70.57%
F1-Score: 0.4033
Log-Loss: 0.5197
              precision    recall  f1-score   support

           0       0.69      0.96      0.80     76508
           1       0.80      0.27      0.40     44779

    accuracy                           0.71    121287
   macro avg       0.75      0.62      0.60    121287
weighted avg       0.73      0.71      0.66    121287



XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
logloss_xgb = log_loss(y_test, y_pred_proba_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

print(f"Accuracy: {accuracy_xgb * 100:.2f}%")
print(f"F1-Score: {f1_xgb:.4f}")
print(f"Log-Loss: {logloss_xgb:.4f}")

print(classification_report(y_test, y_pred_xgb))

Accuracy: 78.59%
F1-Score: 0.7141
Log-Loss: 0.4105
              precision    recall  f1-score   support

           0       0.84      0.82      0.83     76508
           1       0.70      0.72      0.71     44779

    accuracy                           0.79    121287
   macro avg       0.77      0.77      0.77    121287
weighted avg       0.79      0.79      0.79    121287



SVM

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svm_base = LinearSVC(random_state=42, dual=False, max_iter=2000)

svm_model = CalibratedClassifierCV(svm_base, cv=3)

svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)
y_pred_proba_svm = svm_model.predict_proba(X_test)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
logloss_svm = log_loss(y_test, y_pred_proba_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print(f"Accuracy: {accuracy_svm * 100:.2f}%")
print(f"F1-Score: {f1_svm:.4f}")
print(f"Log-Loss: {logloss_svm:.4f}")

print(classification_report(y_test, y_pred_svm))

Accuracy: 76.76%
F1-Score: 0.6676
Log-Loss: 0.4634
              precision    recall  f1-score   support

           0       0.80      0.85      0.82     76508
           1       0.71      0.63      0.67     44779

    accuracy                           0.77    121287
   macro avg       0.75      0.74      0.74    121287
weighted avg       0.76      0.77      0.76    121287

